# CSE281 Digital Image Processing - Project Milestone 1
## Problem Formulation and Image Analysis
**Topic:** Concrete Surface Crack Detection and Structural Damage Assessment
**Course:** CSE281 - Digital Image Processing | Alamein International University (AIU)
---


### 1. Problem Formulation
**Target Users:** Structural engineers, civil infrastructure inspectors, and municipal maintenance squads responsible for auditing bridges, highway pavements, and concrete structures.

**Pipeline Output:** A clean binary crack segmentation mask, geometric damage metrics (cracked surface area percentage, crack length, entity count), and an automated civil safety verdict (**PASS / MONITOR / FAIL**).

**Cost of Wrong Outputs:**
- **False Negatives (Missed Cracks):** High hazard cost. Overlooked cracks allow water and chloride penetration, accelerating rebar corrosion and risking catastrophic structural collapse.
- **False Positives (Surface Pores Mistaken for Cracks):** High economic cost. Unnecessary road closures, structural halting, and costly destructive sonic testing on healthy concrete.

**Primary Metric:** Intersection over Union (**IoU / Jaccard Index**). IoU penalizes both missed crack pixels (false negatives) and spurious noise pixels (false positives) strictly on thin, low-prevalence structural targets where pixel accuracy would be deceptively high (>98% on background).

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath('..'))
from src.data import load_image, load_mask, load_dataset, train_test_split
from src.frequency import compute_dft_spectrum
from src.metrics import iou

print('Loaded DIP exploration libraries!')

### 2. Dataset Exploration & Statistical Profiling
Loading sample images and analyzing size, data types, intensity range, histograms, and frequency spectra.

In [ ]:
images_dir = '../data/raw/images'
masks_dir = '../data/raw/masks'
dataset = load_dataset(images_dir, masks_dir)
print(f'Total dataset size: {len(dataset)} paired images.')

sample = dataset[0]
img = sample['image']
mask = sample['mask']

print(f'Image Dimensions: {img.shape}')
print(f'Data Type:        {img.dtype}')
print(f'Dynamic Range:    [{img.min()}, {img.max()}]')
print(f'Mean Intensity:   {img.mean():.2f}')
print(f'Std Deviation:    {img.std():.2f}')

In [ ]:
# Visualize typical image, difficult low-contrast image, and histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('Typical Concrete Surface (001.jpg)')

axes[0, 1].imshow(mask, cmap='gray')
axes[0, 1].set_title('Ground Truth Crack Mask')

_, spectrum = compute_dft_spectrum(img)
axes[0, 2].imshow(spectrum, cmap='magma')
axes[0, 2].set_title('Centered 2D Fourier Spectrum')

axes[1, 0].hist(img.ravel(), bins=256, range=[0, 256], color='gray')
axes[1, 0].set_title('Intensity Histogram')

difficult_sample = dataset[15]['image']
axes[1, 1].imshow(difficult_sample, cmap='gray')
axes[1, 1].set_title('Difficult Low-Contrast Surface')

axes[1, 2].hist(difficult_sample.ravel(), bins=256, range=[0, 256], color='darkred')
axes[1, 2].set_title('Low-Contrast Narrow Histogram')

plt.tight_layout()
plt.show()

### 3. Spatial Resolution and Gray-Level Reduction Study
**Objective:** Determine the lowest viable spatial resolution and quantization bit depth before crack continuity is destroyed.

In [ ]:
# 1. Gray-Level Quantization (8-bit down to 1-bit)
bit_depths = [8, 6, 4, 2, 1]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for idx, bits in enumerate(bit_depths):
    levels = 2 ** bits
    quantized = np.floor(img.astype(float) / (256.0 / levels)) * (256.0 / levels)
    quantized = np.clip(quantized, 0, 255).astype(np.uint8)
    axes[idx].imshow(quantized, cmap='gray')
    axes[idx].set_title(f'{bits}-Bit ({levels} Levels)')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

# 2. Spatial Subsampling (1x down to 8x downsampling)
scale_factors = [1, 2, 4, 8]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, scale in enumerate(scale_factors):
    h, w = img.shape
    down = cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_AREA)
    up = cv2.resize(down, (w, h), interpolation=cv2.INTER_NEAREST)
    axes[idx].imshow(up, cmap='gray')
    axes[idx].set_title(f'Downsample {scale}x ({w//scale}x{h//scale})')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### 4. Findings & Architectural Design Decisions

| Empirical Finding | Physical Diagnostic Cause | Pipeline Design Decision |
| :--- | :--- | :--- |
| **Narrow Gray-Level Histogram** | Concrete surface weathering and overcast lighting compress dynamic range | Implement **Gamma correction ($\gamma=0.85$)** and **percentile contrast stretching** before thresholding |
| **Uneven Illumination & Shadows** | Directional sunlight and structural overhangs defeat global thresholds | Employ **Local Adaptive Gaussian Thresholding** rather than fixed or global Otsu |
| **High-Frequency Background Granularity** | Sand, gravel, and concrete aggregate produce high-frequency speckle | Apply **2D Butterworth Low-Pass filter ($D_0=45$)** in frequency domain to attenuate speckle without blurring linear edges |
| **Hairline Crack Discontinuities** | Sub-pixel narrowing and thresholding produce broken segments | Apply **Directional Morphological Closing ($5\times 5$ ellipse)** to bridge breaks along crack traces |
| **Isolated Concrete Air Voids (Pores)** | Dark pits mistaken for cracks by adaptive threshold | Use **Area Opening / Connected Object Removal** to prune components smaller than 40 px |
| **Quantization Limit** | Hairline cracks disappear when bit depth drops below 4 bits | Maintain minimum bit depth of **6 bits (64 gray levels)**; 8 bits standard uint8 |
| **Spatial Resolution Limit** | Cracks thinner than 2 pixels vanish beyond 2x downsampling | Maintain original resolution or downsample by at most **2x (240x160 px)** |